In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
from tqdm import tqdm

import tarfile
import os
import scipy.io
import shutil
import json

import warnings
warnings.filterwarnings("ignore", message="The value of the smallest subnormal.*")

In [2]:
import sys
sys.executable

'/mnt/home/the10/netrep-analysis/venv/bin/python'

### 1. Download Data

In [15]:
train_tar = '/mnt/home/gkrawezik/ceph/AI_DATASETS/ImageNet/2012/ILSVRC2012_img_train.tar'
output_dir = '/mnt/home/the10/netrep-analysis/datasets/imagenet'
class_list_file = '/mnt/home/the10/netrep-analysis/datasets/imagenet/imagenet100_classes.json'

# Load class list
with open(class_list_file, 'r') as f:
    target_classes = json.load(f).keys()

# Extract only target class
with tarfile.open(train_tar, 'r') as archive:
    for member in archive.getmembers():
        name = os.path.basename(member.name)
        synset = name.replace('.tar', '')
        if synset in target_classes:
            print(f"✅ Extracting {synset}")
            class_tar_path = archive.extractfile(member)
            class_output_dir = os.path.join(output_dir, synset)
            os.makedirs(class_output_dir, exist_ok=True)
            with tarfile.open(fileobj=class_tar_path) as class_tar:
                class_tar.extractall(path=class_output_dir)

print(f"✅ Extraction complete! Output: {output_dir}")

✅ Extracting n01440764
✅ Extracting n01443537
✅ Extracting n01484850
✅ Extracting n01491361
✅ Extracting n01494475
✅ Extracting n01496331
✅ Extracting n01498041
✅ Extracting n01514668
✅ Extracting n01514859
✅ Extracting n01531178
✅ Extracting n01537544
✅ Extracting n01560419
✅ Extracting n01582220
✅ Extracting n01592084
✅ Extracting n01601694
✅ Extracting n01608432
✅ Extracting n01614925
✅ Extracting n01622779
✅ Extracting n01630670
✅ Extracting n01632458
✅ Extracting n01632777
✅ Extracting n01644900
✅ Extracting n01664065
✅ Extracting n01665541
✅ Extracting n01667114
✅ Extracting n01667778
✅ Extracting n01675722
✅ Extracting n01677366
✅ Extracting n01685808
✅ Extracting n01687978
✅ Extracting n01693334
✅ Extracting n01695060
✅ Extracting n01698640
✅ Extracting n01728572
✅ Extracting n01729322
✅ Extracting n01729977
✅ Extracting n01734418
✅ Extracting n01735189
✅ Extracting n01739381
✅ Extracting n01740131
✅ Extracting n01742172
✅ Extracting n01749939
✅ Extracting n01751748
✅ Extractin

In [56]:
# extract_imagenet100_val.py

val_tar = '/mnt/home/gkrawezik/ceph/AI_DATASETS/ImageNet/2012/ILSVRC2012_img_val.tar'
devkit_tar = '/mnt/home/gkrawezik/ceph/AI_DATASETS/ImageNet/2012/ILSVRC2012_devkit_t12.tar.gz'
output_dir = '/mnt/home/the10/netrep-analysis/datasets/imagenet'
class_list_file = '/mnt/home/the10/netrep-analysis/datasets/imagenet/imagenet100_classes.json'

# Step 1: Load target synsets
with open(class_list_file, 'r') as f:
    target_synsets = list(json.load(f).keys())

# Step 2: Extract devkit + parse mappings
with tarfile.open(devkit_tar, 'r:gz') as dev:
    dev.extractall(path=output_dir)

# Load ground truth (index 1–1000)
gt_path = os.path.join(output_dir, 'ILSVRC2012_devkit_t12/data/ILSVRC2012_validation_ground_truth.txt')
meta_path = os.path.join(output_dir, 'ILSVRC2012_devkit_t12/data/meta.mat')

with open(gt_path, 'r') as f:
    val_labels = [int(line.strip()) for line in f]

# Load synset list from meta.mat
meta = scipy.io.loadmat(meta_path)
synsets = [str(x[0][1][0]) for x in meta['synsets']]
idx_to_synset = {i + 1: synsets[i] for i in range(len(synsets))}

# Step 3: Extract val images & filter
with tarfile.open(val_tar, 'r') as archive:
    members = sorted(archive.getmembers(), key=lambda x: x.name)

    for i, member in enumerate(members):
        synset = idx_to_synset[val_labels[i]]
        if synset in target_synsets:
            dest_dir = os.path.join(output_dir, synset)
            os.makedirs(dest_dir, exist_ok=True)
            filename = f"{synset}_{i:05d}.JPEG"
            dest_path = os.path.join(dest_dir, filename)
            with archive.extractfile(member) as src, open(dest_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)

### 2. Imagenet for training

In [4]:
data_dir = "/mnt/gpuxl/scc/AI_DATASETS/ImageNet/2012/imagenet"
train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "val")
test_dir = os.path.join(data_dir, "test")

# Hyperparameters
batch_size = 4096
num_workers = 48
num_epochs = 90
learning_rate = 0.1

In [5]:
# Data transforms
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])

In [ ]:
# Datasets and loaders
train_dataset = datasets.ImageFolder(train_dir, train_transforms)
val_dataset = datasets.ImageFolder(val_dir, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)

# Model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=None).to(device)
model = torch.nn.DataParallel(model)  # For multi-GPU training

criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

In [5]:
# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {running_loss/len(train_dataset):.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Validation Accuracy: {100 * correct / total:.2f}%")

# Save model
torch.save(model.state_dict(), "resnet18_imagenet.pth")

Epoch 1/90, Training Loss: 3.7903
Validation Accuracy: 17.26%
Epoch 2/90, Training Loss: 2.9778
Validation Accuracy: 32.84%
Epoch 3/90, Training Loss: 2.5360
Validation Accuracy: 39.25%
Epoch 4/90, Training Loss: 2.2404
Validation Accuracy: 43.50%
Epoch 5/90, Training Loss: 2.0082
Validation Accuracy: 49.46%
Epoch 6/90, Training Loss: 1.8362
Validation Accuracy: 51.82%
Epoch 7/90, Training Loss: 1.6970
Validation Accuracy: 55.33%
Epoch 8/90, Training Loss: 1.5877
Validation Accuracy: 57.24%
Epoch 9/90, Training Loss: 1.5012
Validation Accuracy: 59.48%
Epoch 10/90, Training Loss: 1.4252
Validation Accuracy: 58.18%
Epoch 11/90, Training Loss: 1.3620
Validation Accuracy: 60.42%
Epoch 12/90, Training Loss: 1.3104
Validation Accuracy: 63.87%
Epoch 13/90, Training Loss: 1.2708
Validation Accuracy: 63.43%
Epoch 14/90, Training Loss: 1.2311
Validation Accuracy: 64.09%
Epoch 15/90, Training Loss: 1.1958
Validation Accuracy: 64.57%
Epoch 16/90, Training Loss: 1.1643
Validation Accuracy: 65.40%
E

KeyboardInterrupt: 

In [6]:
torch.save(model.state_dict(), "resnet18_imagenet.pth")